### Подготовка признака "описания вакансии"

В рамках данного ноутбука будет проведена обработка описаний вакансий - токенизация и лемматизация

In [ ]:
import pandas as pd

df_full = pd.read_csv('vacancies_merged_prizn.csv')

In [156]:
df_full

,profession,profession_full,vacancy_description,date_published,payment_from,payment_to,experience,employment,city,company_name,company_description,vacancy_count,staff_count,catalogues,source
0,Начальник отдела,Руководитель отдела сопровождения внутренней и...,"Компания ""Санкт-Петербургская биржа"" Вам предс...",2026-03-25,NaN,NaN,NaN,Полная занятость,Москва,Санкт-Петербургская биржа,NaN,NaN,NaN,NaN,scrapping
1,Java,Java (Senior)( ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМ...,"Компания ""ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ""...",2026-03-27,NaN,NaN,Более 6 лет,Полная занятость,Москва,ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ,NaN,NaN,NaN,NaN,scrapping
2,Специалист,Middle+ AQA специалист,"Компания ""Компания Ритейл Сервис"" Мы - аккреди...",2026-03-26,140000.0,180000.0,NaN,Полная занятость,Барнаул,Компания Ритейл Сервис,NaN,NaN,NaN,NaN,scrapping
3,Технолог,Технолог( АКРИХИН ),"Компания ""АКРИХИН"" Обязанности: Обеспечивать ...",2026-03-25,NaN,NaN,NaN,Полная занятость,Старая Купавна,АКРИХИН,NaN,NaN,NaN,NaN,scrapping
4,Project manager,Project Manager Game Dev( Beresnev Games ),"Компания ""Beresnev Games"" Мы в поиске Project ...",2026-03-26,NaN,NaN,NaN,Полная занятость,Москва,Beresnev Games,NaN,NaN,NaN,NaN,scrapping
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30862,Практикант / стажёр Документовед,Практикант / стажёр Документовед,ГНЦ РФ АО «ВНИИНМ» разрабатывает конструкционн...,2026-04-08,0.0,0.0,NaN,NaN,"{'id': 4, 'title': 'Москва', 'declension': 'в ...",Росатом,"Российская государственная корпорация, объедин...",3.0,менее 50,"[{'id': 1, 'title': 'Административная работа, ...",api
30863,Технический специалист/ Системный администратор,Технический специалист/ Системный администратор,"Обязанности:\n• Установка, обновление и админи...",2026-04-08,80000.0,100000.0,NaN,NaN,"{'id': 4, 'title': 'Москва', 'declension': 'в ...",ГБОУ Школа № 657,"Наша Школа - это пространство возможностей, тв...",1.0,менее 50,"[{'id': 33, 'title': 'IT, Интернет, связь, тел...",api
30864,Стажер в IT отдел,Стажер в IT отдел,Здесь занимаются поддержкой IT-платформ и их р...,2026-04-08,0.0,0.0,NaN,NaN,"{'id': 4, 'title': 'Москва', 'declension': 'в ...",Добрый,Ведущий российский бренд безалкогольных напитк...,0.0,более 5000,"[{'id': 33, 'title': 'IT, Интернет, связь, тел...",api
30865,Сотрудник технической поддержки,Сотрудник технической поддержки,"Добрый день!\nМы компания Lasertech, занимаемс...",2026-04-07,80000.0,0.0,NaN,NaN,"{'id': 4, 'title': 'Москва', 'declension': 'в ...",Lasertech,Lasertech — один из ведущих игроков на рынке п...,1.0,менее 50,"[{'id': 33, 'title': 'IT, Интернет, связь, тел...",api


In [157]:
description_column = "vacancy_description"

In [158]:
# в рамках нашей задачи основной датасет это колонка с описанием, поэтому отдельный датасет сделаем
df = df_full[description_column].copy()

#### Подготовка

Нужно убрать из описания все лишнее, кроме слов, которые характеризуют смысл вакансии

Воспользуемся библиотекой spacy https://github.com/explosion/spaCy
Это современное и популярное решение, которое позволяет качественно обрабатывать текст

Мы будем использовать small модель для русского языка, чтобы ускорить обработку

Выстроим пайплайн на примере одного описания

In [159]:
!pip install spacy
!python -m spacy download ru_core_news_sm



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 17.8 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')


In [160]:
import spacy
nlp = spacy.load("ru_core_news_sm")


In [161]:
doc = nlp(df.iloc[0])

doc[0].pos_

'NOUN'

In [162]:
for token in doc[:20]:
    # слово, часть речи, тип именнованной сущности
    print(token, token.pos_, token.ent_type_)


Компания NOUN 
" PUNCT 
Санкт ADJ ORG
- ADJ ORG
Петербургская ADJ ORG
биржа NOUN ORG
" PUNCT 
Вам PRON 
предстоит VERB 
заниматься VERB 
: PUNCT 
  SPACE 
осуществление NOUN 
общего ADJ 
руководства NOUN 
Подразделением NOUN LOC
по ADP 
всем DET 
направлениям NOUN 
его DET 


Полезны только ADJ и NOUN, так же надо удалять именнованные сущности

In [163]:
good_pos = set(['NOUN', 'ADV'])

filtered_tokens = [token.lemma_ for token in doc if token.pos_ in good_pos and not token.ent_type_]

filtered_tokens

['компания',
 'осуществление',
 'руководство',
 'направление',
 'работа',
 'взаимодействие',
 'работник',
 'подразделение',
 'вопрос',
 'разработка',
 'автоматизация',
 'поддержка',
 'система',
 'организация',
 'контроль',
 'разработка',
 'регламент',
 'спецификация',
 'отчёт',
 'осуществление',
 'деятельность',
 'рынок',
 'отчётность',
 'обеспечение',
 'контроль',
 'ведение',
 'соответствие',
 'требование',
 'законодательство',
 'акт',
 'документ',
 'отчётность',
 'организация',
 'контроль',
 'направление',
 'отчёт',
 'осуществление',
 'деятельность',
 'рынок',
 'отчётность',
 'обеспечение',
 'контроль',
 'подтверждение',
 'отправка',
 'отчёт',
 'участие',
 'разработка',
 'форма',
 'отчётность',
 'доработка',
 'отчёт',
 'исправление',
 'ошибка',
 'разработка',
 'отчёт',
 'анализ',
 'контроль',
 'изменение',
 'срок',
 'разработка',
 'отчёт',
 'дата',
 'постановка',
 'задание',
 'контроль',
 'исполнение',
 'задача',
 'тикетах',
 'срок',
 'участие',
 'тестирование',
 'форма',
 'отчёт',
 

Напишем процессинг для всего датасета, применив ранее полученные знания и оптимизации, чтобы быстро обработать 30к строк

#### Процессинг всего датасета

In [164]:
good_pos = set(['NOUN', 'ADV', 'PROPN']) # PROPN часто помечаются названия технологий
good_ent = set(['ORG', '']) # технологиям присваивает ORG или пустоту

# маленькая модель + отключаем парсер, он строит зависимости между словами, они нам не нужны
nlp = spacy.load("ru_core_news_sm", disable=["parser"])


# функция принимает на вход уже програнный через spacy текст и фильтрует его
def text_to_clear_tokens(doc):
    """
    Берем леммы слов (начальные формы), среди тех, 
    чья часть речи в good_pos и ent_type_ (не именновання сущность) нет
    """
    return [token.lemma_ for token in doc if token.pos_ in good_pos and token.ent_type_ in good_ent]

In [165]:
df.isna().sum()

np.int64(3)

In [166]:
# сохраняем индексы, чтобы потом восстановить какая вакансия к чему относится
indexes = df.dropna().index.tolist()
texts = df.dropna().tolist()

In [171]:
import os

from tqdm import tqdm

results = []
result_csv_file = 'vacancy_tokens.csv'

if result_csv_file not in os.listdir('./'):
    
# поточная обработка
# берем по 500 строк и загоняем в модель в кол-во потоков = числу ядер на процессоре
# для визуализации tqdm, внутри передается итератор и он отображает прогресс бар
    for doc in tqdm(nlp.pipe(texts, batch_size=500, n_process=-1), total=len(texts)):
        results.append(text_to_clear_tokens(doc))

In [172]:
results[:5]

[]

In [173]:
if result_csv_file not in os.listdir('./'):
    df_tokens = pd.DataFrame({
        'index': indexes,
        'tokens': [' '.join(tokens) for tokens in results]
    })
    df_tokens.to_csv('vacancy_tokens.csv', index=False)


#### Добавление в основной датасет колонки с токенамизированным описанием

In [174]:
df_tokens = pd.read_csv(result_csv_file, index_col=0)
df_tokens

,tokens
index,
0,компания биржа осуществление руководство напра...
1,компания технологии отраслевой трансформация с...
2,компания компания - ит - компания по artix- ре...
3,компания акрихин обязанность соблюдение персон...
4,компания поиск команда внесете вклад достижени...
...,...
30862,гнц ао вниинм материал топливо вид реактор исс...
30863,обязанность установка обновление администриров...
30864,здесь поддержка it - платформа развитие также ...


In [ ]:
df_full.vacancy_description.isna().sum()

profession                 0
profession_full            0
vacancy_description        3
date_published             0
payment_from           13087
payment_to             19816
experience             27059
employment              6758
city                       0
company_name               0
company_description    24110
vacancy_count          24109
staff_count            24109
catalogues             24109
source                     0
dtype: int64

In [176]:
df_full = df_full.dropna(subset=["vacancy_description"])
df_full.vacancy_description.isna().sum()

np.int64(0)

In [177]:
df_full["vacancy_description_tokenize"] = df_tokens['tokens']

In [178]:
df_full[['vacancy_description','vacancy_description_tokenize']]

,vacancy_description,vacancy_description_tokenize
0,"Компания ""Санкт-Петербургская биржа"" Вам предс...",компания биржа осуществление руководство напра...
1,"Компания ""ТЕХНОЛОГИИ ОТРАСЛЕВОЙ ТРАНСФОРМАЦИИ""...",компания технологии отраслевой трансформация с...
2,"Компания ""Компания Ритейл Сервис"" Мы - аккреди...",компания компания - ит - компания по artix- ре...
3,"Компания ""АКРИХИН"" Обязанности: Обеспечивать ...",компания акрихин обязанность соблюдение персон...
4,"Компания ""Beresnev Games"" Мы в поиске Project ...",компания поиск команда внесете вклад достижени...
...,...,...
30862,ГНЦ РФ АО «ВНИИНМ» разрабатывает конструкционн...,гнц ао вниинм материал топливо вид реактор исс...
30863,"Обязанности:\n• Установка, обновление и админи...",обязанность установка обновление администриров...
30864,Здесь занимаются поддержкой IT-платформ и их р...,здесь поддержка it - платформа развитие также ...
30865,"Добрый день!\nМы компания Lasertech, занимаемс...",день компания lasertech производство продажа о...


Все ок совпадает, сохраняем

In [179]:
df_full.to_csv('vacancies_merged_prizn_tokenize.csv', index=False)